# BanglaBERT (`csebuetnlp/banglabert_small`): emotions and topic

Pre-trained encoder with two heads: 6 sigmoid outputs for emotions (multi-label)
and 12 softmax outputs for topic (single-label multiclass).

Reproducibility notes:

* The Hub revision is **pinned** to commit `a9507c8f05866c6786f4fbfeeb582f72c90d6ba1`,
  so a re-run fetches exactly these weights. Before pinning, `from_pretrained`
  silently tracked the `main` branch and would return different weights later.
* `csebuetnlp/banglabert_small` is licensed **CC BY-NC-SA 4.0** (non-commercial,
  share-alike); see `README.md`.
* Protocol is shared with the other notebooks through `evaluation.py`: a
  70/15/15 multi-label-stratified split, early stopping on the validation split,
  and a single final test evaluation.

Run with `PYTHONHASHSEED=42`. Metrics land in `results/bert_*.json`.


In [ ]:
import os

import numpy as np
import pandas as pd
import tensorflow as tf
from bnlp import CleanText
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from transformers import AutoTokenizer, TFAutoModel

from evaluation import (
    EMOTIONS,
    SEED,
    best_thresholds,
    emotions_metrics,
    majority_baseline,
    save_metrics,
    set_global_seed,
    split_indices,
    topic_metrics,
)

os.makedirs("results", exist_ok=True)
set_global_seed(SEED)


## Pinned checkpoint

`BANGLABERT_REVISION` is a commit SHA, not a branch name. Record it in any
results you publish so the weights are traceable.


In [ ]:
BANGLABERT_REPO = "csebuetnlp/banglabert_small"
BANGLABERT_REVISION = "a9507c8f05866c6786f4fbfeeb582f72c90d6ba1"
BANGLABERT_LICENSE = "cc-by-nc-sa-4.0"

tokenizer = AutoTokenizer.from_pretrained(BANGLABERT_REPO, revision=BANGLABERT_REVISION)
bert_model = TFAutoModel.from_pretrained(
    BANGLABERT_REPO, revision=BANGLABERT_REVISION, from_pt=True
)
print("loaded", BANGLABERT_REPO, "at", BANGLABERT_REVISION)
print("license:", BANGLABERT_LICENSE)


In [ ]:
df = pd.read_csv("Bangla Emotion Dataset.csv")

clean_text = CleanText(
    fix_unicode=True,
    unicode_norm=True,
    unicode_norm_form="NFKC",
    remove_url=True,
    remove_email=True,
    remove_emoji=True,
    remove_number=True,
    remove_digits=True,
    remove_punct=True,
    replace_with_url="",
    replace_with_email="",
    replace_with_number="",
    replace_with_digit="",
    replace_with_punct="",
)

df["clean_description"] = df["Data"].apply(clean_text)

y_emotions = df[EMOTIONS].to_numpy(dtype=int)
topic_encoder = LabelEncoder()
y_topic = topic_encoder.fit_transform(df["Topic"])

splits = split_indices(y_emotions, seed=SEED)
train_idx, val_idx, test_idx = splits["train"], splits["val"], splits["test"]
print({name: len(idx) for name, idx in splits.items()})


## Tokenize each split separately


In [ ]:
max_len = 128
# The Functional model below declares exactly these inputs. A BERT tokenizer
# also returns `token_type_ids`; it is dropped here so the dict fed to
# fit/predict matches the declared inputs (single-segment text, so the
# encoder's default all-zero segment ids are equivalent).
MODEL_INPUTS = ("input_ids", "attention_mask")


def tokenize_function(texts):
    tokens = tokenizer(
        list(texts),
        padding="max_length",
        truncation=True,
        max_length=max_len,
        return_tensors="tf",
    )
    return {name: tokens[name] for name in MODEL_INPUTS}


train_texts = df["clean_description"].iloc[train_idx]
val_texts = df["clean_description"].iloc[val_idx]
test_texts = df["clean_description"].iloc[test_idx]

train_tokens = tokenize_function(train_texts)
val_tokens = tokenize_function(val_texts)
test_tokens = tokenize_function(test_texts)

emotions_train, emotions_val, emotions_test = (
    y_emotions[train_idx],
    y_emotions[val_idx],
    y_emotions[test_idx],
)
topic_train, topic_val, topic_test = (
    y_topic[train_idx],
    y_topic[val_idx],
    y_topic[test_idx],
)

train_dataset = tf.data.Dataset.from_tensor_slices(
    (train_tokens, {"emotions": emotions_train, "topics": topic_train})
).batch(16)
val_dataset = tf.data.Dataset.from_tensor_slices(
    (val_tokens, {"emotions": emotions_val, "topics": topic_val})
).batch(16)
test_dataset = tf.data.Dataset.from_tensor_slices(
    (test_tokens, {"emotions": emotions_test, "topics": topic_test})
).batch(16)

baseline = majority_baseline(emotions_test)


In [ ]:
def build_model(num_emotions, num_topics):
    input_ids = Input(shape=(max_len,), dtype=tf.int32, name="input_ids")
    attention_mask = Input(shape=(max_len,), dtype=tf.int32, name="attention_mask")

    bert_output = bert_model(input_ids, attention_mask=attention_mask)[0]
    cls_token = bert_output[:, 0, :]

    emotions_output = Dense(num_emotions, activation="sigmoid", name="emotions")(cls_token)
    topics_output = Dense(num_topics, activation="softmax", name="topics")(cls_token)

    model = Model(inputs={"input_ids": input_ids, "attention_mask": attention_mask},
                  outputs=[emotions_output, topics_output])
    model.compile(
        optimizer=Adam(learning_rate=2e-5),
        loss={"emotions": "binary_crossentropy", "topics": "sparse_categorical_crossentropy"},
        metrics={"emotions": ["accuracy"], "topics": ["accuracy"]},
    )
    return model


model = build_model(len(EMOTIONS), len(topic_encoder.classes_))
model.summary()


## Training

Early stopping on validation loss; the restored weights are the ones evaluated on
the test split.


In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss", patience=1, restore_best_weights=True, verbose=1
)

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=3,
    callbacks=[early_stopping],
    verbose=2,
)


## Final test evaluation

Predictions are decoded explicitly, so metric names and their definitions are
attached to the values rather than inferred from `model.evaluate` positions.


In [ ]:
val_emotion_scores, val_topic_scores = model.predict(val_dataset)
thresholds = best_thresholds(emotions_val, val_emotion_scores)
print("thresholds from validation:", dict(zip(EMOTIONS, np.round(thresholds, 2))))

test_emotion_scores, test_topic_scores = model.predict(test_dataset)

emotion_results = emotions_metrics(emotions_test, (test_emotion_scores >= thresholds).astype(int))
emotion_results["model"] = f"banglabert_small@{BANGLABERT_REVISION[:8]} (CLS head)"
emotion_results["split"] = "test 15%, unseen during training and threshold selection"
emotion_results["thresholds"] = dict(zip(EMOTIONS, thresholds.tolist()))
emotion_results["majority_baseline"] = baseline

topic_results = topic_metrics(topic_test, test_topic_scores.argmax(axis=1), list(topic_encoder.classes_))
topic_results["model"] = f"banglabert_small@{BANGLABERT_REVISION[:8]} (CLS head)"
topic_results["split"] = "test 15%, unseen during training"

print("emotions macro F1:", round(emotion_results["macro_f1"], 4))
print("emotions subset accuracy:", round(emotion_results["subset_accuracy"], 4))
print("emotions hamming loss:", round(emotion_results["hamming_loss"], 4))
print("model per-label accuracy:", round(emotion_results["per_label_accuracy"], 4))
print("majority per-label accuracy:", round(baseline["per_label_majority_accuracy"], 4))
print("topic accuracy:", round(topic_results["accuracy"], 4))
print("topic macro F1:", round(topic_results["macro_f1"], 4))


In [ ]:
model.save_weights("results/banglabert_weights.weights.h5")
tokenizer.save_pretrained("results/banglabert_tokenizer")

save_metrics("results/banglabert_emotions.json", emotion_results)
save_metrics("results/banglabert_topic.json", topic_results)
save_metrics("results/banglabert_baseline.json", baseline)
print("saved results/banglabert_*.json and weights")


## Predictions on held-out text


In [ ]:
def predict_emotion_and_topic(text):
    tokens = tokenize_function([clean_text(text)])
    emotion_scores, topic_scores = model.predict(tokens, verbose=0)
    emotion_scores = emotion_scores[0]
    labels = [name for name, score, thr in zip(EMOTIONS, emotion_scores, thresholds) if score >= thr]
    topic = topic_encoder.inverse_transform([int(topic_scores[0].argmax())])[0]
    return {
        "Emotions": dict(zip(EMOTIONS, np.round(emotion_scores, 4))),
        "Predicted labels": labels,
        "Topic": topic,
    }


held_out_index = int(test_idx[0])
print("held-out row:", df.loc[held_out_index, "Data"][:80])
print("gold:", [name for name in EMOTIONS if df.loc[held_out_index, name] == 1], df.loc[held_out_index, "Topic"])
print("prediction:", predict_emotion_and_topic(df.loc[held_out_index, "Data"]))
